In [ ]:

# Load best saved model

model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()
print(' Best model loaded (epoch 12, Mean Dice: 0.4244)')

In [ ]:

# RESULT 1 & 2: Full validation — per-class + mean Dice

dice_metric_eval = DiceMetric(
    include_background=False,
    reduction='mean_batch'   # returns per-class scores
)

post_pred_eval  = AsDiscrete(argmax=True,  to_onehot=4)
post_label_eval = AsDiscrete(to_onehot=4)

model.eval()
print('Running full validation set evaluation...')

with torch.no_grad():
    for i, val_data in enumerate(val_loader):
        val_inputs = val_data['image'].to(device)
        val_labels = val_data['label'].to(device)

        val_outputs = sliding_window_inference(
            val_inputs,
            roi_size=(96, 96, 96),
            sw_batch_size=1,
            predictor=model,
            overlap=0.5
        )

        val_outputs_post = [post_pred_eval(o) for o in val_outputs]
        val_labels_post  = [post_label_eval(l) for l in val_labels]

        dice_metric_eval(
            y_pred=torch.stack(val_outputs_post),
            y=torch.stack(val_labels_post)
        )

        if (i + 1) % 10 == 0:
            print(f'  Processed {i+1}/{len(val_loader)} cases...')

# Aggregate
per_class_dice = dice_metric_eval.aggregate()   # shape: [3] for classes 1,2,3
dice_metric_eval.reset()

class_names = ['NCR/NET (Class 1)', 'Edema (Class 2)', 'ET (Class 3)']
mean_dice_final = per_class_dice.mean().item()

print('\n' + '=' * 55)
print('   FINAL VALIDATION RESULTS — SwinUNETR BraTS2020')
print('=' * 55)
for name, score in zip(class_names, per_class_dice):
    print(f'  {name:<22}: {score.item():.4f}')
print('-' * 55)
print(f'  Mean Dice (foreground) : {mean_dice_final:.4f}')
print('=' * 55)

In [ ]:

# RESULT 2: Print nicely formatted class-wise Dice table

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

class_scores = [s.item() for s in per_class_dice]

table_data = [
    ['NCR/NET (Class 1)', f'{class_scores[0]:.4f}'],
    ['Edema   (Class 2)', f'{class_scores[1]:.4f}'],
    ['ET      (Class 3)', f'{class_scores[2]:.4f}'],
    ['Mean Dice', f'{mean_dice_final:.4f}'],
]

col_labels = ['Segmentation Class', 'Dice Score']
tbl = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.5, 0.25]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(13)
tbl.scale(1, 2)

# Style header row
for j in range(2):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Highlight mean row
for j in range(2):
    tbl[4, j].set_facecolor('#27ae60')
    tbl[4, j].set_text_props(color='white', fontweight='bold')

plt.title('SwinUNETR — Class-wise Dice Scores (BraTS 2020)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dice_table.png'), dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: dice_table.png')

In [ ]:

# RESULT 3 & 4: Training Loss Curve + Validation Dice Curve


train_loss_history = [
    1.4242, 1.0152, 0.8440, 0.7280, 0.6578,
    0.5991, 0.5630, 0.5275, 0.5021, 0.4815,
    0.4604, 0.4469, 0.4414, 0.4332, 0.4263
]
val_dice_epochs = [2, 4, 6, 8, 10, 12, 14]
val_dice_scores = [0.1666, 0.3673, 0.3908, 0.3992, 0.3980, 0.4244, 0.4196]
best_epoch      = 12
best_dice_val   = 0.4244


epochs = list(range(1, len(train_loss_history) + 1))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('SwinUNETR Training — BraTS 2020', fontsize=15, fontweight='bold')


ax1 = axes[0]
ax1.plot(epochs, train_loss_history, 'b-o', markersize=5, linewidth=2, label='Training Loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('DiceCE Loss', fontsize=12)
ax1.set_title('Training Loss Curve', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# Annotate final value
ax1.annotate(f'{train_loss_history[-1]:.4f}',
             xy=(epochs[-1], train_loss_history[-1]),
             xytext=(-30, 10), textcoords='offset points',
             fontsize=10, color='blue',
             arrowprops=dict(arrowstyle='->', color='blue', lw=1.2))


ax2 = axes[1]
ax2.plot(val_dice_epochs, val_dice_scores, 'r-s', markersize=7, linewidth=2, label='Val Mean Dice')
ax2.axvline(x=best_epoch, color='green', linestyle='--', linewidth=1.5, label=f'Best Epoch ({best_epoch})')
ax2.axhline(y=best_dice_val, color='green', linestyle=':', linewidth=1.5, alpha=0.6)
ax2.scatter([best_epoch], [best_dice_val], color='gold', s=120, zorder=5,
            edgecolors='green', linewidths=2, label=f'Best Dice: {best_dice_val:.4f}')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Mean Dice Score', fontsize=12)
ax2.set_title('Validation Dice Curve', fontsize=13)
ax2.set_ylim(0, 0.65)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: training_curves.png')

In [ ]:

# RESULT 5: Visual Slice Comparison
# 4 MRI channels | Ground Truth | Prediction


# Color map: 0=BG(black), 1=NCR/NET(red), 2=Edema(lime), 3=ET(yellow)
seg_cmap = mcolors.ListedColormap(['black', 'red', 'lime', 'yellow'])

model.eval()
with torch.no_grad():
    sample = next(iter(val_loader))
    inputs = sample['image'].to(device)   # (1, 4, H, W, D)
    labels = sample['label'].to(device)   # (1, 1, H, W, D)

    outputs = sliding_window_inference(
        inputs, roi_size=(96, 96, 96),
        sw_batch_size=1, predictor=model, overlap=0.5
    )
    preds = torch.argmax(outputs, dim=1, keepdim=True)  # (1, 1, H, W, D)

# CPU & numpy
inp_np   = inputs[0].cpu().numpy()        # (4, H, W, D)
gt_np    = labels[0, 0].cpu().numpy()     # (H, W, D)
pred_np  = preds[0, 0].cpu().numpy()      # (H, W, D)

# Pick the slice with the most segmentation
tumor_per_slice = (gt_np > 0).sum(axis=(0, 1))
best_z = int(np.argmax(tumor_per_slice))

modality_names = ['T1', 'T1CE', 'T2', 'FLAIR']

fig = plt.figure(figsize=(20, 8))
fig.suptitle(f'BraTS2020 — SwinUNETR Segmentation  |  Slice z={best_z}',
             fontsize=15, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 4, hspace=0.12, wspace=0.06)

# Row 1: 4 MRI modalities
for i in range(4):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(inp_np[i, :, :, best_z].T, cmap='gray', origin='lower')
    ax.set_title(modality_names[i], fontsize=12, fontweight='bold')
    ax.axis('off')

# Row 2 left: GT overlay on FLAIR
ax_gt = fig.add_subplot(gs[1, 0:2])
ax_gt.imshow(inp_np[3, :, :, best_z].T, cmap='gray', origin='lower')
ax_gt.imshow(gt_np[:, :, best_z].T, cmap=seg_cmap, alpha=0.55,
             vmin=0, vmax=3, origin='lower')
ax_gt.set_title('Ground Truth', fontsize=13, fontweight='bold')
ax_gt.axis('off')

# Row 2 right: Prediction overlay on FLAIR
ax_pred = fig.add_subplot(gs[1, 2:4])
ax_pred.imshow(inp_np[3, :, :, best_z].T, cmap='gray', origin='lower')
ax_pred.imshow(pred_np[:, :, best_z].T, cmap=seg_cmap, alpha=0.55,
               vmin=0, vmax=3, origin='lower')
ax_pred.set_title('SwinUNETR Prediction', fontsize=13, fontweight='bold')
ax_pred.axis('off')

# Legend
legend_patches = [
    mpatches.Patch(color='black',  label='Background'),
    mpatches.Patch(color='red',    label='NCR/NET (Class 1)'),
    mpatches.Patch(color='lime',   label='Edema (Class 2)'),
    mpatches.Patch(color='yellow', label='Enhancing Tumor (Class 3)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.04),
           fancybox=True, edgecolor='gray')

plt.savefig(os.path.join(RESULTS_DIR, 'visual_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: visual_comparison.png')

In [ ]:

# RESULT 6: Scrolling Slice GIF
# Shows GT vs Prediction across all axial slices


gif_path = os.path.join(RESULTS_DIR, 'segmentation_scroll.gif')
frames   = []

# Only render slices where there is some content
active_slices = [z for z in range(0, gt_np.shape[2], 2)
                 if inp_np[3, :, :, z].max() > 0]

print(f'Generating GIF from {len(active_slices)} slices...')

for z in active_slices:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(inp_np[3, :, :, z].T, cmap='gray', origin='lower')
    axes[0].set_title(f'FLAIR  z={z}', fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(inp_np[3, :, :, z].T, cmap='gray', origin='lower')
    axes[1].imshow(gt_np[:, :, z].T, cmap=seg_cmap, alpha=0.5, vmin=0, vmax=3, origin='lower')
    axes[1].set_title('Ground Truth', fontsize=10)
    axes[1].axis('off')

    axes[2].imshow(inp_np[3, :, :, z].T, cmap='gray', origin='lower')
    axes[2].imshow(pred_np[:, :, z].T, cmap=seg_cmap, alpha=0.5, vmin=0, vmax=3, origin='lower')
    axes[2].set_title('Prediction', fontsize=10)
    axes[2].axis('off')

    plt.tight_layout()
    fig.canvas.draw()
    frame_arr = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    frame_arr = frame_arr.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame_arr)
    plt.close(fig)

imageio.mimsave(gif_path, frames, fps=6)
print(f' Saved GIF: {gif_path}')
display(IPImage(filename=gif_path, width=700))